In [1]:
%pip install -q google-genai

StatementMeta(, 2685bc1b-6472-406b-b364-cd189713af57, 7, Finished, Available, Finished, True)


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /nfs4/pyenv-8abafe2c-8884-461f-a44f-aa9c9df446c8/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [2]:
import json, time
from PIL import Image
from google import genai

# Clave leída desde OneLake (nunca escrita en el código)
api_key = notebookutils.fs.head("Files/config/gemini_api_key.txt").strip()
client = genai.Client(api_key=api_key)

# Mismo modelo que uso en la detección
MODELO = "gemini-3.5-flash-lite"
PAUSA_SEG = 5
TABLA_SALIDA = "Lh_Silver.silver_detecciones"

StatementMeta(, 2685bc1b-6472-406b-b364-cd189713af57, 9, Finished, Available, Finished, False)

In [3]:
# Experimento: ¿sabe Gemini contar paneles? Lo pruebo con la planta terminada (2022),
# donde conozco el total real: 1.430.000 paneles (dato de Iberdrola)
PROMPT_CONTEO = """Esta es una imagen aérea (ortofoto) de unos 300 x 300 metros de una planta fotovoltaica,
con una resolución de unos 0,3 m por píxel. Cada panel solar mide aproximadamente 2 x 1 metros.
Estima cuántos paneles solares individuales hay en la imagen.
Responde solo con un JSON así: {"num_paneles": 0}"""

# Teselas que forman la planta en 2022 (las mismas que uso en Gold como zona de planta)
zona = (spark.table(TABLA_SALIDA).filter("anio = 2022 AND pct_paneles >= 5")
        .join(spark.table("bronze_teselas"), ["anio", "tesela"])
        .select("tesela", "ruta").collect())

conteos = []
for t in zona:
    imagen = Image.open(f"/lakehouse/default/{t.ruta}").convert("RGB")
    try:
        r = client.models.generate_content(model=MODELO, contents=[imagen, PROMPT_CONTEO],
                                           config={"response_mime_type": "application/json"})
        resp = json.loads(r.text)
        if isinstance(resp, list):
            resp = resp[0]
        conteos.append(int(resp["num_paneles"]))
    except Exception as e:
        print("  Error:", str(e)[:100])
        if "429" in str(e):
            break
    time.sleep(PAUSA_SEG)

# Si no ha dado tiempo a todas, extrapolo con la media por tesela
total_est = sum(conteos) / len(conteos) * len(zona)
print(f"Teselas contadas: {len(conteos)} de {len(zona)}")
print(f"Paneles por tesela (media): {sum(conteos) / len(conteos):,.0f}")
print(f"Total estimado por Gemini: {total_est:,.0f}")
print(f"Total real del proyecto:   1,430,000")
print(f"Error: {(total_est - 1_430_000) / 1_430_000 * 100:+.1f} %")

StatementMeta(, 2685bc1b-6472-406b-b364-cd189713af57, 10, Finished, Available, Finished, False)

Teselas contadas: 102 de 102
Paneles por tesela (media): 3,699
Total estimado por Gemini: 377,285
Total real del proyecto:   1,430,000
Error: -73.6 %
